In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import desc, count

In [4]:
spark = SparkSession.builder \
    .appName("EcommerceProductAnalysis") \
    .master("local[*]") \
    .getOrCreate()

In [5]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("ecommerce.csv")

df.show()

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7.65|     17850|United Kingdom|
|   536365|    21730|GLASS STAR FROSTE...|       6|12/1/2010 8:26|     4.

In [6]:
most_expensive = df.orderBy(desc("UnitPrice"))

most_expensive.show(1)

+---------+---------+-----------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+-----------+--------+---------------+---------+----------+--------------+
|  C556445|        M|     Manual|      -1|6/10/2011 15:31|  38970.0|     15098|United Kingdom|
+---------+---------+-----------+--------+---------------+---------+----------+--------------+
only showing top 1 row


In [7]:
grouped = df.groupBy("Country").count()

grouped.show()

+---------------+-----+
|        Country|count|
+---------------+-----+
|         Sweden|  462|
|      Singapore|  229|
|        Germany| 9495|
|         France| 8557|
|         Greece|  146|
|        Belgium| 2069|
|        Finland|  695|
|          Italy|  803|
|           EIRE| 8196|
|      Lithuania|   35|
|         Norway| 1086|
|          Spain| 2533|
|        Denmark|  389|
|      Hong Kong|  288|
|        Iceland|  182|
|         Israel|  297|
|Channel Islands|  758|
|         Cyprus|  622|
|    Switzerland| 2002|
|        Lebanon|   45|
+---------------+-----+
only showing top 20 rows


In [8]:
country_count = df.groupBy("Country") \
    .agg(count("*").alias("total_products"))

country_count.show()

+---------------+--------------+
|        Country|total_products|
+---------------+--------------+
|         Sweden|           462|
|      Singapore|           229|
|        Germany|          9495|
|         France|          8557|
|         Greece|           146|
|        Belgium|          2069|
|        Finland|           695|
|          Italy|           803|
|           EIRE|          8196|
|      Lithuania|            35|
|         Norway|          1086|
|          Spain|          2533|
|        Denmark|           389|
|      Hong Kong|           288|
|        Iceland|           182|
|         Israel|           297|
|Channel Islands|           758|
|         Cyprus|           622|
|    Switzerland|          2002|
|        Lebanon|            45|
+---------------+--------------+
only showing top 20 rows


In [9]:
country_count.toPandas().to_csv(
    "country_count.csv",
    index=False
)

most_expensive.limit(1).toPandas().to_csv(
    "most_expensive_product.csv",
    index=False
)